### Import Libraries

In [1]:
# Get the absolute path to the project root
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

import torch
import mlflow
import logging
from datetime import datetime
from pathlib import Path
from torch.utils.data import DataLoader
from src import (UNet, ResidualUNet, AttentionUNetV3, FeaturePyramidUNet, FeedbackResUNet, TransformerUNet,
                 train_model, weights_init, evaluate_dice_score, visualize_predictions)
from dataloader import ACDCDataset
import matplotlib.pyplot as plt
%matplotlib inline

INFO:albumentations.check_version:A new version of Albumentations is available: 2.0.5 (you have 1.4.7). Upgrade using: pip install --upgrade albumentations


### Setup Experiment

In [2]:
# Setup logging
dataset = 'img_slices_org_v2'
model_name = 'transformer_unet'
exp_prefix = f'{model_name}_{dataset}'
exp_run_id = f'{exp_prefix}_{datetime.now().strftime("%Y%m%d%H%M%S")}'
log_file_name = f'./logs/{exp_run_id}.log'
logging.basicConfig(filename=log_file_name, level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', force=True)

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logging.info(f'Using device {device}')
print(f'Using device {device}')

# Setup MLflow
# mlflow.login() # host: https://community.cloud.databricks.com/
# mlflow.set_tracking_uri("databricks")
# mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")
# %env MLFLOW_TRACKING_URI=sqlite:///outputs/cmri_seg_mlruns.db
# mlflow.set_experiment("/cmri-segmentation-of-ventricular-structures-and-myocardium")

# mlflow ui --port 8080 --backend-store-uri sqlite:///outputs/cmri_seg_mlruns.db
# tensorboard --logdir=./logs/tensorboard_runs/

Using device cuda


In [3]:
dir_checkpoint = Path(f'./models/checkpoints/{exp_run_id}/')

epochs = 100
batch_size = 32
lr = 5e-4
scale = 1
amp = False
class_weights = torch.tensor([0.3, 0.4, 0.3]).to(device=device)

### Data Loading

In [4]:
root_dir = r'../data/ACDC/{}/'.format(dataset)
logging.info(f'Using root_dir {root_dir}')

training_dataset = ACDCDataset(root_dir=root_dir, dataset='training')
validation_dataset = ACDCDataset(root_dir=root_dir, dataset='validation')
testing_dataset = ACDCDataset(root_dir=root_dir, dataset='testing')

logging.info(f'Training dataset size: {len(training_dataset)}')
logging.info(f'Validation dataset size: {len(validation_dataset)}')
logging.info(f'Testing dataset size: {len(testing_dataset)}')

# Load data from the dataset
train_dataloader = DataLoader(training_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(testing_dataset, batch_size=batch_size, shuffle=False)

torch.manual_seed(18)

Loaded 2084 training images
Loaded 260 validation images
Loaded 260 testing images


In [ ]:
# Iterate through the dataset and plot the first 4 samples
n_samples_to_plot = 2

for i, sample in enumerate(test_dataloader):
    if i >= n_samples_to_plot:
        break

    # Extract data from the sample
    org_image = sample['org_image'][i].squeeze().numpy()
    image = sample['image'][i].squeeze().numpy()
    org_msk_all = sample['org_masks_all'][i].squeeze().numpy()
    msk_all = sample['masks_all'][i].squeeze().numpy()

    # Create a figure with subplots
    fig, axes = plt.subplots(1, 4, figsize=(8, 5))

    # Plot the image and masks
    axes[0].imshow(org_image, cmap='gray')
    axes[0].set_title('Org. Image')
    axes[0].axis('off')

    axes[1].imshow(org_msk_all, cmap='gray')
    axes[1].set_title('Org. Combined Mask')
    axes[1].axis('off')

    axes[2].imshow(image, cmap='gray')
    axes[2].set_title('Image')
    axes[2].axis('off')

    axes[3].imshow(msk_all, cmap='gray')
    axes[3].set_title('Combined Mask')
    axes[3].axis('off')

    # Adjust layout and show the plot
    plt.tight_layout()
    plt.show()

### Model Training

In [ ]:
model = TransformerUNet(in_channels=1, num_classes=4)

# Setup trainer
multi_class = True
use_checkpoints = False

if use_checkpoints:
    checkpoint_epoch = 1
    checkpoint_path = str(dir_checkpoint / f'checkpoint_epoch{checkpoint_epoch}.pth')
    checkpoint = torch.load(checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint)
    start_epoch = checkpoint_epoch + 1
else:
    start_epoch = 1
    model.apply(weights_init)

model = model.to(memory_format=torch.channels_last)

logging.info(f'Network:\n'
                 f'\t{model.n_channels} input channels\n'
                 f'\t{model.n_classes} output channels (classes)\n'
                 f'\t{"Bilinear" if model.bilinear else "Transposed conv"} upscaling')

model.to(device=device)

try:
    run_id = train_model(
        dataset=dataset,
        model=model,
        training_dataset=training_dataset,
        validation_dataset=validation_dataset,
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        epochs=epochs,
        start_epoch=start_epoch,
        batch_size=batch_size,
        learning_rate=lr,
        device=device,
        model_name=model_name,
        img_scale=scale,
        amp=amp,
        exp_run_id=exp_run_id,
        multi_class=multi_class,
        dir_checkpoint=dir_checkpoint,
    )
except torch.cuda.OutOfMemoryError:
    logging.error('Detected OutOfMemoryError! '
                  'Enabling checkpointing to reduce memory usage, but this slows down training. '
                  'Consider enabling AMP (--amp) for fast and memory efficient training')
    torch.cuda.empty_cache()
    model.use_checkpointing()
    run_id = train_model(
        dataset=dataset,
        model=model,
        training_dataset=training_dataset,
        validation_dataset=validation_dataset,
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        epochs=epochs,
        start_epoch=start_epoch,
        batch_size=batch_size,
        learning_rate=lr,
        device=device,
        model_name=model_name,
        img_scale=scale,
        amp=amp,
        exp_run_id=exp_run_id,
        multi_class=multi_class,
        dir_checkpoint=dir_checkpoint,
    )

### Model Evaluation

In [ ]:
# Model Evaluation Using a Checkpoint
model = ResidualUNet(in_channels=1, out_channels=4)

# Load the state dict from the checkpoint
checkpoint_dir = Path(f'./models/checkpoints/{exp_run_id}/')
checkpoint = torch.load(str(checkpoint_dir / 'checkpoint_epoch{}.pth'.format('7')))
model.load_state_dict(checkpoint)
model = model.to(device)
model.eval()


In [ ]:
# Assuming you have a DataLoader, model, and device
visualize_predictions(dataloader=test_dataloader, model=model, device=torch.device('cuda'), num_samples=3)

In [ ]:
# run_id = "f4d2e0deb89f458ca522161566e6e949"

with mlflow.start_run(run_id=run_id):
    # Compute the Dice score for each class in the Train set
    avg_dice_scores_batch = evaluate_dice_score(model, train_dataloader, device=device)
    print("Dice Scores - Train Set:")
    print(f"BKG: {avg_dice_scores_batch[0]:.4f}\tLV: {avg_dice_scores_batch[1]:.4f}\tRV: {avg_dice_scores_batch[2]:.4f}\tMYO: {avg_dice_scores_batch[3]:.4f}\tOVR: {round(torch.tensor(avg_dice_scores_batch).mean().item(), 4)}")
    # print(f"LV: {avg_dice_scores_batch[0]:.4f}\tRV: {avg_dice_scores_batch[1]:.4f}\tMYO: {avg_dice_scores_batch[2]:.4f}\tOVR: {round(torch.tensor(avg_dice_scores_batch).mean().item(), 4)}")
    mlflow.log_metric("train_dc_bkg", round(avg_dice_scores_batch[0].item(), 4))
    mlflow.log_metric("train_dc_lv", round(avg_dice_scores_batch[1].item(), 4))
    mlflow.log_metric("train_dc_rv", round(avg_dice_scores_batch[2].item(), 4))
    mlflow.log_metric("train_dc_myo", round(avg_dice_scores_batch[3].item(), 4))
    mlflow.log_metric("train_dice_score", round(torch.tensor(avg_dice_scores_batch).mean().item(), 4)) # avg_dice_scores_batch[1:]


    # Compute the Dice score for each class in the Validation set
    avg_dice_scores_batch = evaluate_dice_score(model, val_dataloader, device=device)
    print("\nDice Scores - Validation Set:")
    print(f"BKG: {avg_dice_scores_batch[0]:.4f}\tLV: {avg_dice_scores_batch[1]:.4f}\tRV: {avg_dice_scores_batch[2]:.4f}\tMYO: {avg_dice_scores_batch[3]:.4f}\tOVR: {round(torch.tensor(avg_dice_scores_batch).mean().item(), 4)}")
    # print(f"LV: {avg_dice_scores_batch[0]:.4f}\tRV: {avg_dice_scores_batch[1]:.4f}\tMYO: {avg_dice_scores_batch[2]:.4f}\tOVR: {round(torch.tensor(avg_dice_scores_batch).mean().item(), 4)}")
    mlflow.log_metric("val_dc_bkg", round(avg_dice_scores_batch[0].item(), 4))
    mlflow.log_metric("val_dc_lv", round(avg_dice_scores_batch[1].item(), 4))
    mlflow.log_metric("val_dc_rv", round(avg_dice_scores_batch[2].item(), 4))
    mlflow.log_metric("val_dc_myo", round(avg_dice_scores_batch[3].item(), 4))
    mlflow.log_metric("val_dice_score", round(torch.tensor(avg_dice_scores_batch).mean().item(), 4))


    # Compute the Dice score for each class in the Test set
    avg_dice_scores_batch = evaluate_dice_score(model, test_dataloader, device=device)
    print("\nDice Scores - Test Set:")
    print(f"BKG: {avg_dice_scores_batch[0]:.4f}\tLV: {avg_dice_scores_batch[1]:.4f}\tRV: {avg_dice_scores_batch[2]:.4f}\tMYO: {avg_dice_scores_batch[3]:.4f}\tOVR: {round(torch.tensor(avg_dice_scores_batch).mean().item(), 4)}")
    # print(f"LV: {avg_dice_scores_batch[0]:.4f}\tRV: {avg_dice_scores_batch[1]:.4f}\tMYO: {avg_dice_scores_batch[2]:.4f}\tOVR: {round(torch.tensor(avg_dice_scores_batch).mean().item(), 4)}")
    mlflow.log_metric("test_dc_bkg", round(avg_dice_scores_batch[0].item(), 4))
    mlflow.log_metric("test_dc_lv", round(avg_dice_scores_batch[1].item(), 4))
    mlflow.log_metric("test_dc_rv", round(avg_dice_scores_batch[2].item(), 4))
    mlflow.log_metric("test_dc_myo", round(avg_dice_scores_batch[3].item(), 4))
    mlflow.log_metric("test_dice_score", round(torch.tensor(avg_dice_scores_batch).mean().item(), 4))